# FeedbackLens: Exploratory Data Analysis & NLP Pipeline
This notebook demonstrates the end-to-end data pipeline for customer review intelligence.

In [ ]:
import csv
import re
import json
from collections import Counter
from pathlib import Path


## 1. Load Sample Reviews Dataset

In [ ]:
csv_path = Path('../examples/sample_reviews.csv')
with open(csv_path, 'r', encoding='utf-8') as f:
    reviews = list(csv.DictReader(f))
print(f'Loaded {len(reviews)} reviews.')
reviews[:2]

## 2. PII Redaction Demonstration
Before sending customer feedback to an external LLM API, PII (email, phone numbers) must be masked.

In [ ]:
def redact_pii(text: str) -> str:
    text = re.sub(r'\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b', '[EMAIL]', text, flags=re.I)
    return re.sub(r'(?<!\w)(?:\+?\d[\d\s().-]{7,}\d)(?!\w)', '[PHONE]', text)

sample = 'Contact me at user@example.com or +1 415 555 0199 for my order.'
print('Original:', sample)
print('Redacted:', redact_pii(sample))

## 3. Gemini Structured JSON Schema & Model Config

In [ ]:
with open('../config.json') as f:
    config = json.load(f)
print('Configured Model:', config['model'])
print('Timeout:', config['timeout_seconds'], 'seconds')

## 4. Aggregate Report Generation (Deterministic Python Step)

In [ ]:
mock_results = [
    {'id': '1', 'analysis': {'sentiment': 'positive', 'priority': 'low', 'aspects': [{'name': 'product_quality', 'sentiment': 'positive'}]}},
    {'id': '2', 'analysis': {'sentiment': 'negative', 'priority': 'high', 'aspects': [{'name': 'support', 'sentiment': 'negative'}]}},
    {'id': '3', 'analysis': {'sentiment': 'mixed', 'priority': 'medium', 'aspects': [{'name': 'usability', 'sentiment': 'positive'}, {'name': 'pricing', 'sentiment': 'negative'}]}},
]

sentiment_counts = Counter(r['analysis']['sentiment'] for r in mock_results)
priority_counts = Counter(r['analysis']['priority'] for r in mock_results)
aspect_counts = Counter(a['name'] for r in mock_results for a in r['analysis']['aspects'])

print('Sentiment Breakdown:', dict(sentiment_counts))
print('Priority Breakdown:', dict(priority_counts))
print('Aspect Mentions:', dict(aspect_counts))